# Regional Indicator Gold

## Target
`pondolens.dev_gold.fact_regional_indicator`

## Grain and key
One row per `year_id + region_id + indicator_id + breakdown_type + breakdown_value + price_basis`. `fact_indicator_id` is the surrogate primary key and starts at **50000**. Existing observations keep their IDs when values are updated.

## Repeatable processing
`INCREMENTAL` merges new and changed observations. `FULL_SNAPSHOT` replaces only explicitly listed complete source slices. An unchanged rerun makes no business-data changes.

This notebook reads validated Silver tables; it does not ingest the raw files in the shared Drive folder. Confirm source table names and dimension IDs before running. A successful Databricks execution and validation are still required before production use.

## BP and AP coverage
- **BP:** Provides regional need/development context for comparing with government spending.
- **AP1:** Regional indicator context alongside expenditure by agency/region.
- **AP2:** Does not calculate appropriation or obligation gaps; use the regional expenditure fact for those measures.
- **AP3:** Supports comparison of obligations with GRDP per capita, unemployment, and water, internet, and electricity access.

Indicators are contextual measures. They do not establish that spending caused an indicator outcome. Keep units, price basis, breakdowns, and missing values intact.

- Keeps GRDP with price_basis = 2018 only
- Keeps unemployment rate.
- Adds Service Access using the correct measure_type column.
- Excludes national-level Philippines observations.
- Matches the existing dim_indicator values, including N/A, Total, and All.


In [0]:
%sql

CREATE OR REPLACE TABLE pondolens.dev_gold.fact_regional_indicator
USING DELTA
AS

WITH all_indicators AS (

    -- 1. GRDP PER CAPITA: 2018 PRICE BASIS ONLY
    SELECT
        CAST(year AS INT) AS observation_year,
        region_code,

        'grdp' AS indicator,
        'grdp_per_capita' AS measure_type,
        CAST(grdp_per_capita AS DECIMAL(20,8)) AS measurement,

        'pesos' AS unit,
        TRIM(CAST(price_basis AS STRING)) AS price_basis,
        'n/a' AS service,
        'total' AS breakdown_type,
        'all' AS breakdown_value

    FROM pondolens.dev_silver.grdp_per_capita

    WHERE LOWER(TRIM(geographic_level)) = 'region'
      AND TRIM(CAST(price_basis AS STRING)) = '2018'
      AND grdp_per_capita IS NOT NULL

    UNION ALL

    -- 2. UNEMPLOYMENT RATE
    SELECT
        CAST(year AS INT) AS observation_year,
        region_code,

        'unemployment' AS indicator,
        'unemployment_rate' AS measure_type,
        CAST(unemployment_rate AS DECIMAL(20,8)) AS measurement,

        'percent' AS unit,
        'n/a' AS price_basis,
        'n/a' AS service,
        'total' AS breakdown_type,
        'all' AS breakdown_value

    FROM pondolens.dev_silver.unemployment_rate

    WHERE unemployment_rate IS NOT NULL

    UNION ALL

    -- 3. SERVICE ACCESS
    SELECT
        CAST(year AS INT) AS observation_year,
        region_code,

        'service_access' AS indicator,
        LOWER(TRIM(service_measure)) AS measure_type,
        CAST(value AS DECIMAL(20,8)) AS measurement,

        LOWER(TRIM(unit)) AS unit,
        'n/a' AS price_basis,
        LOWER(TRIM(service)) AS service,
        LOWER(TRIM(breakdown_type)) AS breakdown_type,
        LOWER(TRIM(breakdown_value)) AS breakdown_value

    FROM pondolens.dev_silver.service_access_available

    WHERE value IS NOT NULL
      AND COALESCE(UPPER(TRIM(region_code)), '') <> 'PHL'
      AND COALESCE(UPPER(TRIM(region)), '') <> 'PHILIPPINES'
),

mapped_indicators AS (

    SELECT
        y.year_id,
        r.region_id,
        d.indicator_id,
        s.indicator,
        s.measurement

    FROM all_indicators s

    INNER JOIN pondolens.dev_gold.dim_year y
        ON s.observation_year = y.year

    INNER JOIN pondolens.dev_gold.dim_region r
        ON REGEXP_REPLACE(
            UPPER(TRIM(s.region_code)),
            '[-\\s]',
            ''
        ) = REGEXP_REPLACE(
            UPPER(TRIM(r.region_code)),
            '[-\\s]',
            ''
        )
       AND LOWER(TRIM(r.geographic_level)) = 'region'

    INNER JOIN pondolens.dev_gold.dim_indicator d
        ON LOWER(TRIM(s.indicator)) = LOWER(TRIM(d.indicator))
       AND LOWER(TRIM(s.measure_type)) = LOWER(TRIM(d.measure_type))

       AND LOWER(TRIM(s.unit))
           <=> LOWER(TRIM(d.unit))

       AND LOWER(TRIM(s.price_basis))
           <=> LOWER(TRIM(d.price_basis))

       AND LOWER(TRIM(s.service))
           <=> LOWER(TRIM(d.service))

       AND LOWER(TRIM(s.breakdown_type))
           <=> LOWER(TRIM(d.breakdown_type))

       AND LOWER(TRIM(s.breakdown_value))
           <=> LOWER(TRIM(d.breakdown_value))
),

deduplicated AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY year_id, region_id, indicator_id
            ORDER BY measurement DESC
        ) AS rn

    FROM mapped_indicators
)

SELECT
    50000 + ROW_NUMBER() OVER (
        ORDER BY year_id, region_id, indicator_id
    ) AS fact_indicator_id,

    year_id,
    indicator_id,
    region_id,
    indicator,
    measurement

FROM deduplicated

WHERE rn = 1;


In [0]:
%sql
-- Validate that all three indicators are present
SELECT
    indicator,
    indicator_id,
    COUNT(*) AS total_records,
    COUNT(DISTINCT region_id) AS total_regions,
    COUNT(DISTINCT year_id) AS total_years

FROM pondolens.dev_gold.fact_regional_indicator

GROUP BY indicator, indicator_id

ORDER BY indicator, indicator_id;

## Run notes
- Schedule with Databricks Job **maximum concurrent runs = 1** and prevent other writers from concurrently changing this target.
- Confirm the three Silver table names, mapped column names, and `dim_indicator` attributes match your workspace.
- Keep national observations distinct from the 17 regions; do not infer regional values.
- Preserve measurement units and price basis. Do not sum percentages or substitute one year's value for missing years.
- Regional comparisons provide context; they do not prove that spending caused an indicator outcome.